# Semenov et al. (2026) on dandiset 001515

Paper. Semenov I, Silkuniene G, Silkunas M, Pakhomova ON, Bixler JN, Ibey BL, Kochunov P, Pakhomov AG
(2026). Sub-microsecond optical measurements of cell membrane charging and lesioning by pulsed electric
fields. *Bioelectrochemistry* 171, 109283.
[doi:10.1016/j.bioelechem.2026.109283](https://doi.org/10.1016/j.bioelechem.2026.109283). The claims file
(`claims/001515/10.1016-j.bioelechem.2026.109283.yaml`) was written from the version of record (main
text, Methods, Results, Discussion and the figure legends; the paper names no supplement and links no
code).

Dandiset. [DANDI:001515](https://dandiarchive.org/dandiset/001515/0.261006.1841), version
0.261006.1841, "Plasma Membrane Polarization Dynamics of CHO Cells During 1 us Pulses: Effects of Cell
Size and Electric Field Strength". The paper cites no dataset ("Data will be made available on
request"). This is the only published version, and the most recent; the draft holds the same files
(compared below), so the choice cannot change a result.

What the dandiset holds. 292 NWB files from 37 subjects in four sessions (2023-05-08 to
2023-10-10). Each holds one 70-frame strobe image series of a field of CHO cells loaded with FluoVolt,
exposed to single 1-us pulses at one amplitude (20 to 100 V across electrodes 1.00 mm apart), recorded
"forward" or "backward", with the anode above or below. The data check below shows that the frames
sample the time course of membrane charging and discharging (the frame order runs backward in the
"backward" files). This is the time-course protocol the paper attributes to its reference [20]
(Semenov et al., Bioelectrochemistry 168 (2025) 109143; Discussion: "similar to what we have reported
recently using weaker, non-electroporating PEFs [20]"). None of this paper's experiments is in it: no
sets of 40 baseline and 60 pulse images at 900 ns, no field escalation within a cell, no Di-8-ANEPPS,
0.3-us pulses, Gd3+, two-pulse or Fluo-4 recordings. The files do not record the delay of each frame
or the field strength in V/cm, and the electrode spacing (1.00 mm) differs from the paper's (1.6 mm).

Claims. 34 claims. One, `fig5b-conductivity-calibration`, is a computation from the paper's
Eq. (2) and uses no recordings; it is tested. The other 33 are blocked, `data_missing`.

| Claim | Tested | Reason if blocked |
|---|---|---|
| fig5b-conductivity-calibration | yes | |
| fig2a-fluovolt-saturation, fig2a-fluovolt-stable, fig3-dye-sensitivity, fig3-fluovolt-nonlinear, fig4a-fluovolt-limits, fig4a-fluovolt-cathodic-decline | no | data_missing: the FluoVolt sets of 40 + 60 images at 900 ns with field escalation are not in the dandiset |
| fig1-schwan-pattern, fig1-pole-dimming, fig2b-di8-climbs-to-570, fig2b-di8-bleaching, fig2b-di8-decline-within-set, fig2cd-stable-before-electroporation, fig2c-1000-within-set, fig2d-lasting-reduction, fig4b-di8-limits, fig4b-first-10-pulses, fig4b-cathodic-asymmetry, fig4b-poles-alike-570-780, fig4c-equator-follows-cosine | no | data_missing: no Di-8-ANEPPS recordings |
| fig4d-short-pulses-same-limit | no | data_missing: no 0.3-us pulses |
| fig5a-gd-extends-anodic, fig5a-gd-linear, fig5a-gd-cathodic, fig5a-gd-differences, fig5c-gd-reduces-conductivity | no | data_missing: no Gd3+ or parallel control series |
| fig6a-ca-single-pulse, fig6b-ca-train-responders, fig6b-ca-650 | no | data_missing: no Fluo-4 Ca2+ imaging |
| fig7a-cosine-profiles, fig7b-charging-during-pulse, fig7ab-test-pulse-recovery, fig7c-fast-conductivity-drop, fig7c-full-reseal-mild | no | data_missing: no two-pulse recordings or Di-8-ANEPPS escalation |

In [ ]:
import collections
import time
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dandi.dandiapi import DandiAPIClient

import strobe

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)

started = time.time()
assets = strobe.assets()
with ProcessPoolExecutor(6) as pool:
    meta = pd.DataFrame(list(pool.map(strobe.read_metadata, assets)))
print(f"{len(assets)} NWB files at dandiset {strobe.DANDISET_ID} version {strobe.DANDISET_VERSION}; "
      f"metadata read in {time.time() - started:.0f} s")

## Data check

First the archive's metadata, the draft against the pinned version, and everything one file holds.

In [ ]:
m = strobe.version_metadata()
print("Title:", m["name"])
print("Description:", m["description"])
print("Related resources:", [(r.get("relation"), r.get("identifier")) for r in m.get("relatedResource", [])])
summary = m["assetsSummary"]
print("Assets summary: files", summary["numberOfFiles"], "subjects", summary["numberOfSubjects"],
      "species", [s["name"] for s in summary.get("species", [])],
      "approach", summary.get("approach"), "measurement", summary.get("measurementTechnique"))

with DandiAPIClient() as client:
    draft = sorted((a.path, a.size) for a in client.get_dandiset(strobe.DANDISET_ID, "draft").get_assets())
pinned = sorted((a["path"], a["size"]) for a in assets)
print("\nDraft and pinned version hold the same paths and sizes:", draft == pinned, f"({len(draft)} and {len(pinned)} files)")

one = meta.iloc[0]
print("\nEverything in", one.path, "(apart from the NWB specifications):")
for g in one.groups:
    print("  ", g)
for k in ["subject_id", "species", "age", "strain", "session_id", "session_start", "shape", "dtype", "unit",
          "conversion", "rate", "starting_time", "has_timestamps", "dimension", "acquisition", "stimulus",
          "processing", "analysis", "intervals"]:
    print(f"  {k}: {one[k]}")
print("\nImageSeries description:", one.series_description)
print("\nSession description:", one.session_description)

Every file holds one image series and nothing else: no stimulus series, no trial or epoch table, no
processed data, no per-frame timestamps (only a start time and the stated 6 Hz), and no field strength
or laser delay. The species in the archive summary, "Cricetulus griseus - Cricetulus aureus", names two
species for one CHO cell line (the files give NCBITaxon_10029, *C. griseus*), and the subject age "P1D"
is given for a cell line.

In [ ]:
print("Values of the metadata fields over the 292 files:")
for k in ["shape", "dtype", "unit", "rate", "has_timestamps", "pulse_us", "spacing_mm", "electrode_diameter_mm",
          "dye", "sine", "acquisition", "stimulus", "processing", "intervals", "species", "age"]:
    print(f"  {k}:", dict(collections.Counter(map(str, meta[k]))))
print("  ImageSeries descriptions (first 80 characters):",
      dict(collections.Counter(d[:80] for d in meta.series_description)))

meta["session"] = meta.session_id.str.replace("session_", "")
counts = (meta.groupby(["session", "subject"])
          .agg(files=("path", "size"), volts=("volts", lambda v: sorted(set(v))),
               orders=("order", lambda v: dict(collections.Counter(v))),
               anode=("anode", lambda v: dict(collections.Counter(v))),
               starts=("session_start", lambda v: sorted({s[:10] for s in v}))))
print("\nFiles per subject:")
print(counts.to_string())
print("\nSubjects per session:", meta.groupby("session").subject.nunique().to_dict())
print("Files per amplitude (V):", meta.volts.value_counts().sort_index().to_dict())
print("Subjects per amplitude (V):", meta.groupby("volts").subject.nunique().to_dict())

The paper's Methods 2.2 to 2.3 describe electrodes 1.6 mm apart, field strengths in V/cm from Sim4Life
modelling, flashes at 900 ns (1-us pulses) or 275 ns (0.3-us pulses), and sets of 100 images (40 without
the pulse, then 60 with it), with the amplitude escalated set by set within a cell; two dyes,
FluoVolt and Di-8-ANEPPS. The files hold none of that structure: 70 frames per file, one amplitude per
subject (20 to 100 V), electrodes 1.00 mm apart, FluoVolt only, and no record of what distinguishes
the two files of each subject that share an order and an anode side. One subject
(CHO-20230925-c1) has 4 files where the others have 8. The series description mentions "sine wave
exposures"; no file describes one.

Next every file's pixels are read once (about 2 minutes): a checksum of every frame, the frame
brightness, and a membrane polarization index per frame (the module docstring of `strobe.py` gives
its parameters, standard choices for this check, not the paper's).

In [ ]:
started = time.time()
with ProcessPoolExecutor(6) as pool:
    summaries = list(pool.map(strobe.summarize_file, assets))
print(f"pixels of {len(summaries)} files read in {time.time() - started:.0f} s")
px = pd.DataFrame([{k: s[k] for k in ["path", "n_cells", "baseline", "peak", "peak_frame", "first_half",
                                      "last_half", "n_half", "min", "max"]} for s in summaries])
px["frame0_ratio"] = [s["frame_mean"][0] / np.median(s["frame_mean"][1:]) for s in summaries]
d = meta.merge(px, on="path")
print("Pixel values over all files: min", d["min"].min(), "max", d["max"].max(), "(uint16, unit n.a.)")
print("Cells found per file (connected parts of the membrane mask of at least", strobe.MIN_CELL_PX, "px):",
      d.n_cells.describe()[["min", "50%", "max"]].to_dict())

where = collections.defaultdict(list)
for s in summaries:
    for i, h in enumerate(s["frame_md5"]):
        where[h].append((s["path"], i))
shared = [v for v in where.values() if len(v) > 1]
print("\nFrames whose pixel values occur in more than one place:", len(shared), "of", sum(len(v) for v in where.values()))

X = np.array([s["mean_image_small"].ravel() for s in summaries])
X = (X - X.mean(1, keepdims=True)) / X.std(1, keepdims=True)
C = X @ X.T / X.shape[1]
subj = d.subject.to_numpy()
same = subj[:, None] == subj[None, :]
np.fill_diagonal(same, False)
other = subj[:, None] != subj[None, :]
within = np.array([np.median(C[i, same[i]]) for i in range(len(d))])
between = np.array([C[i, other[i]].max() for i in range(len(d))])
print("Correlation of each file's mean image (64 x 64) with the other files of its subject, median:",
      f"median over files {np.median(within):.3f}, lowest {within.min():.3f}")
print("Highest correlation with any file of another subject:", f"median {np.median(between):.3f}, max {between.max():.3f}")
odd = d[within < 0.9]
print("Files whose field of view differs from the rest of their subject:")
print(odd[["path", "session_start", "order", "anode", "peak", "n_cells"]].to_string(index=False))

No frame is deposited twice and no field of view appears under two subjects (the highest correlation of
mean images across subjects is printed above, against about 0.99 within a subject). One file of
subject CHO-20230926-c1 shows a different field of view and a session start of 2024-12-16, fourteen
months after the other files of its subject; it is not a recording of that subject's cells.

In [ ]:
print("Frame 0 brightness relative to the median frame (frame 0 / median of frames 1-69):")
print(d.frame0_ratio.describe()[["min", "25%", "50%", "75%", "max"]].round(3).to_dict())
print("Files with frame 0 below 0.8 of the median frame:", int((d.frame0_ratio < 0.8).sum()), "of", len(d))

print("\nPolarization index (top minus bottom of each cell, %), frame of its peak and the frames where it")
print("exceeds half its peak change, by recording order (frame 0 left out):")
print(d.groupby("order")[["peak_frame", "first_half", "last_half", "n_half"]].median().to_string())
fw = d[d.order == "forward"]
bw = d[d.order == "backward"]
print(f"\nForward: half-peak frames {fw.first_half.median():.0f} to {fw.last_half.median():.0f}. "
      f"Backward, read in reverse (69 - frame): {69 - bw.last_half.median():.0f} to {69 - bw.first_half.median():.0f}.")
print("Half-peak width, frames: forward median", fw.n_half.median(), "backward median", bw.n_half.median())

The polarization index rises and falls once in each file, over about 19 frames at half its peak.
In "forward" files this happens at frames 16 to 34, and in "backward" files at the same frames counted
from the end, so the backward recordings hold the same time course in reverse order, as the session
descriptions say. The frames are therefore successive laser delays relative to the pulse (the series
description: a flash "at a precise time interval before, during, or after" the pulse), but the files
do not record the delays. If the 19 frames of half-peak response span the 1-us pulse, the step is
about 50 ns and the 70 frames cover about 3.5 us; this is consistent with the data, not shown by
them. Which frame is 900 ns into the pulse, the time point every claim of the paper uses, cannot be
read from the files. In most files frame 0 is about half as bright as the others.

Next the sign of the index against the anode side the session description gives, and its size
against the pulse amplitude.

In [ ]:
d["sign"] = np.sign(d.peak).astype(int)
table = d.pivot_table(index=["session", "subject"], columns="anode", values="sign",
                      aggfunc=lambda v: " ".join(f"{x:+d}" for x in v))
print("Sign of the peak index (+: top of the cells brighter) per file, by the stated anode side:")
print(table.to_string())

def majority(v):
    return int(np.sign(v.sum())) if v.sum() != 0 else 0
groups = d.groupby(["subject", "anode"]).sign
d["group_sign"] = groups.transform(majority)
against = d[(d.sign != d.group_sign) & (d.group_sign != 0)]
print("\nFiles whose sign differs from the other files of their subject with the same stated anode side:",
      len(against))
print(against[["path", "order", "anode", "peak"]].round(1).to_string(index=False))
flip = d.groupby("subject").apply(lambda g: (majority(g[g.anode == "above"].sign), majority(g[g.anode == "below"].sign)),
                                  include_groups=False)
print("\nSubjects where the majority sign is opposite for the two stated anode sides:",
      int(sum(a == -b != 0 for a, b in flip)), "of", len(flip))
above_sign = d[d.anode == "above"].groupby("session").sign.apply(lambda v: dict(collections.Counter(v)))
print("Sign of the index in files stated 'anode above', per session:", above_sign.to_dict())

In 35 of 37 subjects the sign of the polarization flips with the stated anode side, as it should; 9
files have the sign of the other anode side of their subject (listed above). Across sessions the stated side does not fix the sign: files with "anode
above" are brighter at the top in one session and at the bottom in others, and in the 2023-10-10
session the sign changes between subjects. The image orientation relative to the electrodes is not
stated, so the cathode-facing pole of a cell cannot be located from the metadata alone, only from the
sign of its response (depolarization brightens FluoVolt, Methods 2.3).

In [ ]:
amp = d.groupby(["session", "volts"]).peak.apply(lambda v: np.abs(v).median()).rename("median |peak index| %")
print("Size of the response by session and pulse amplitude:")
print(amp.round(1).to_string())
s10 = d[d.session == "2023-10-10"]
rho = s10.groupby("volts").peak.apply(lambda v: np.abs(v).median())
print("\n2023-10-10 (one subject per amplitude): Spearman rho of median |peak| with volts =",
      round(pd.Series(rho.values).corr(pd.Series(rho.index), method="spearman"), 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
cmap = plt.get_cmap("Blues")
vals = sorted(s10.volts.unique())
lo, hi = np.inf, -np.inf
for ax, order in zip(axes, ["forward", "backward"]):
    for v in vals:
        rows = s10[(s10.volts == v) & (s10.order == order)]
        traces = [summaries[i]["index"] * np.sign(d.loc[i, "peak"]) for i in rows.index]
        tr = np.median(traces, axis=0)[1:]
        lo, hi = min(lo, tr.min()), max(hi, tr.max())
        ax.plot(np.arange(1, 70), tr, color=cmap(0.3 + 0.7 * vals.index(v) / (len(vals) - 1)), lw=1.5, label=f"{v:.0f} V")
    ax.set_title(f"2023-10-10, {order} files")
    ax.set_xlabel("frame")
axes[0].set_ylabel("polarization index, % (sign-aligned)")
axes[1].legend(fontsize=7, frameon=False, ncol=2)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
print(f"Plotted range of the median traces: {lo:.1f} to {hi:.1f} %; frame 0 not plotted; no point outside the axes.")

In the 2023-10-10 session, one subject per amplitude, the response grows with the pulse amplitude
(Spearman rho 1.0 over nine amplitudes), which is consistent with the stated amplitudes. At the same
40 V it differs between sessions (median 25.9% on 2023-05-08, 11.9% on 2023-09-26, 12.8% on
2023-10-10), so amplitude alone does not set the response across sessions.
The amplitudes are in volts across the electrodes; the paper's field strengths in V/cm come from a
Sim4Life model of its 1.6-mm electrode geometry, so they cannot be assigned to these files.

## fig5b-conductivity-calibration

> By Eq. (2), the effective membrane conductivity barely affects charging while it stays far below
> cytosol and medium conductivities; a 10-fold rise reduces IMP by about 2% and a 100-fold rise by 12%.

Criterion: Eq. (2) with the constants of Methods 2.3 (lambda_i 0.4 S/m, lambda_o 1.64 S/m, R 8 um,
d 5 nm, intact lambda_m 3e-7 S/m); the reduction relative to IMPC under readings (a) 1 - f/1.5 and
(b) 1 - f(lambda_m)/f(3e-7 S/m). Expected 2 +- 0.2% (10-fold) and 12 +- 1.2% (100-fold); reproduced
when both are within tolerance under one reading. The computation uses no recordings.

In [ ]:
c = strobe.CONSTANTS
lm0 = c["lambda_m_intact"]
expected = {10: (2.0, 0.2), 100: (12.0, 1.2)}
rows = []
for fold in [1, 10, 100, 1000]:
    f = strobe.f_factor(fold * lm0)
    rows.append({"fold": fold, "lambda_m S/m": fold * lm0, "f": f,
                 "reduction_a_%": 100 * (1 - f / 1.5),
                 "reduction_b_%": 100 * (1 - f / strobe.f_factor(lm0))})
calib = pd.DataFrame(rows)
print("Constants (Methods 2.3):", c)
print(calib.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
for fold, (value, tol) in expected.items():
    row = calib[calib.fold == fold].iloc[0]
    for reading in ["a", "b"]:
        got = row[f"reduction_{reading}_%"]
        print(f"{fold}-fold, reading ({reading}): {got:.2f}% against {value} +- {tol}: "
              f"{'within' if abs(got - value) <= tol else 'outside'} tolerance")

A variant, not part of the criterion: how the 10-fold reduction depends on the two constants the paper
states as approximate (R "the approximate average value of 8 um", d "assumed as 5 nm"), and which
values would give 2%.

In [ ]:
grid = []
for R in [5e-6, 6e-6, 8e-6, 10e-6, 12e-6, 15e-6]:
    for dd in [3e-9, 4e-9, 5e-9, 7e-9]:
        f10, f1 = strobe.f_factor(10 * lm0, R=R, d=dd), strobe.f_factor(lm0, R=R, d=dd)
        f100 = strobe.f_factor(100 * lm0, R=R, d=dd)
        grid.append({"R_um": R * 1e6, "d_nm": dd * 1e9, "10x_a_%": 100 * (1 - f10 / 1.5), "10x_b_%": 100 * (1 - f10 / f1),
                     "100x_a_%": 100 * (1 - f100 / 1.5)})
grid = pd.DataFrame(grid)
print(grid.pivot(index="R_um", columns="d_nm", values="10x_a_%").round(2).to_string())
print("\n100-fold, reading (a):")
print(grid.pivot(index="R_um", columns="d_nm", values="100x_a_%").round(1).to_string())
both = grid[(grid["10x_a_%"].between(1.8, 2.2)) & (grid["100x_a_%"].between(10.8, 13.2))]
print("\nCombinations in this grid with both values within tolerance under reading (a):", len(both))
print(both.round(2).to_string(index=False))

**Result: partially reproduced.** With the paper's constants, Eq. (2) gives a 1.33% (reading a) or
1.20% (reading b) reduction of IMP for a 10-fold rise of membrane conductivity, outside 2 +- 0.2%, and
11.87% or 11.75% for a 100-fold rise, inside 12 +- 1.2%. The qualitative statement, that conductivity far
below the cytosol and medium values barely affects charging, holds: a 1000-fold rise is needed to
reduce IMP by more than half (57.4%). In the grid of radii and membrane thicknesses printed above, no
combination puts both values within tolerance, so the ~2% is not explained by the approximate
constants alone.

## Blocked claims

The other 33 claims need recordings that are not in the dandiset (header table). The check above shows
what is there instead: FluoVolt time courses at one amplitude per cell. Even for the FluoVolt claims
(Fig. 2A, 3 and 4A), the dandiset lacks the escalation sets of 40 baseline and 60 pulse images at
900 ns; and to use its time courses in their place would need the delay of each frame and the field
strength in V/cm, neither of which the files record.

## Summary

| Claim | Outcome |
|---|---|
| fig5b-conductivity-calibration | partially reproduced (10-fold value 1.33% against ~2%; 100-fold value 11.87% against 12%) |
| the other 33 claims | blocked, data_missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's. Only a computation from Eq. (2) was run, with the
paper's constants; no recording of the paper's experiments is in the dandiset.

What was found about the dataset.

- It names this paper (IsPublishedIn) but holds a different experiment: FluoVolt time courses of
  charging and discharging at one pulse amplitude per cell, the protocol the paper attributes to its
  reference [20]. None of the paper's 40 + 60 image sets at 900 ns, field escalations, Di-8-ANEPPS,
  0.3-us, Gd3+, two-pulse or Ca2+ recordings is in it.
- The laser delay of each frame is not recorded; the data show that the frames are successive delays,
  reversed in "backward" files.
- Pulses are given in volts with electrodes 1.00 mm apart, where the paper uses 1.6 mm and modelled
  field strengths in V/cm.
- Within a subject, two files share each combination of order and anode side, with nothing to tell
  them apart. One subject has 4 files.
- One file of CHO-20230926-c1 shows another field of view and was recorded on 2024-12-16.
- The stated anode side does not fix the sign of the response across sessions, and a few files
  disagree with the other files of their subject.
- Frame 0 is about half as bright as the rest in most files.
- The archive summary names two species for a CHO cell line, and the subjects (cell fields) have an
  age of "P1D".